In [3]:
import json
import re
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig
from datasets import load_from_disk
from collections import defaultdict
import os

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [4]:
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
dataset = load_from_disk(f"{dataset_based_path}/mh-function-calling-dataset-hf-v3")
test_dataset = dataset["test"]
print(f"✅ Loaded test dataset with {len(test_dataset)} samples")
print(f"📋 Dataset features: {test_dataset.features}")

✅ Loaded test dataset with 765 samples
📋 Dataset features: {'patient_query': Value('string'), 'type': Value('string'), 'thinking': Value('string'), 'num_days': Value('int64'), 'function': Value('string'), 'patient_id': Value('string'), 'assistant_response': Value('string'), 'text': Value('string')}


In [5]:
peft_model_id = "/home/jovyan/Works/MSc_Thesis/mh_function_calling/model_training/finetuned_models/llama3.2_3b_mh_function_calling_v6"

print("Loading trained model...")

# Load PEFT config
config = PeftConfig.from_pretrained(peft_model_id)
print(f"Base model: {config.base_model_name_or_path}")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map={"": 0},
    torch_dtype=torch.bfloat16
)

# Load tokenizer (with special tokens)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Resize embeddings
model.resize_token_embeddings(len(tokenizer))

# Load adapter weights
model = PeftModel.from_pretrained(model, peft_model_id)
model.to(torch.bfloat16)
model.eval()

print("✅ Model loaded successfully!")

Loading trained model...
Base model: meta-llama/Llama-3.2-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.44it/s]
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Tokenizer vocab size: 128260


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:693: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


✅ Model loaded successfully!


In [6]:
def extract_function_from_response(response):
    try:
        # Strategy 1: Standard extraction with proper tags
        function_match = re.search(r'<function>\s*(.*?)\s*</function>', response, re.DOTALL)
        
        if function_match:
            function_content = function_match.group(1).strip()
            function_data = json.loads(function_content)
            
            return {
                'function_name': function_data.get('name', ''),
                'parameters': function_data.get('parameters', {}),
                'raw_function': function_content,
                'extracted_successfully': True,
                'extraction_method': 'standard'
            }
    except (json.JSONDecodeError, KeyError, AttributeError):
        pass
    
    try:
        # Strategy 2: Look for JSON pattern anywhere in response
        json_pattern = r'"name"\s*:\s*"([^"]+)"\s*,\s*"parameters"\s*:\s*(\{[^}]*\})'
        json_match = re.search(json_pattern, response, re.DOTALL)
        
        if json_match:
            function_name = json_match.group(1)
            params_str = json_match.group(2)
            parameters = json.loads(params_str)
            
            return {
                'function_name': function_name,
                'parameters': parameters,
                'raw_function': json_match.group(0),
                'extracted_successfully': True,
                'extraction_method': 'json_pattern'
            }
    except (json.JSONDecodeError, AttributeError):
        pass
    
    try:
        # Strategy 3: Handle malformed tags but with JSON content
        malformed_patterns = [
            r'<think>\s*\{\s*"name"\s*:\s*"([^"]+)"\s*,\s*"parameters"\s*:\s*(\{[^}]*\})\s*\}\s*</function>',
            r'<function>\s*\{\s*"name"\s*:\s*"([^"]+)"\s*,\s*"parameters"\s*:\s*(\{[^}]*\})\s*\}\s*</think>',
        ]
        
        for pattern in malformed_patterns:
            malformed_match = re.search(pattern, response, re.DOTALL)
            if malformed_match:
                function_name = malformed_match.group(1)
                params_str = malformed_match.group(2)
                parameters = json.loads(params_str)
                
                return {
                    'function_name': function_name,
                    'parameters': parameters,
                    'raw_function': malformed_match.group(0),
                    'extracted_successfully': True,
                    'extraction_method': 'malformed_tags'
                }
    except (json.JSONDecodeError, AttributeError):
        pass
    
    try:
        # Strategy 4: Extract just function name and parameters separately
        function_name = ''
        parameters = {}
        
        # Look for function name pattern
        name_match = re.search(r'"name"\s*:\s*"([^"]+)"', response)
        if name_match:
            function_name = name_match.group(1)
        
        # Look for patient_id
        patient_match = re.search(r'"patient_id"\s*:\s*"([^"]+)"', response)
        if patient_match:
            parameters['patient_id'] = patient_match.group(1)
        
        # Look for num_days
        days_match = re.search(r'"num_days"\s*:\s*(\d+)', response)
        if days_match:
            parameters['num_days'] = int(days_match.group(1))
        
        if function_name:
            return {
                'function_name': function_name,
                'parameters': parameters,
                'raw_function': response,
                'extracted_successfully': True,
                'extraction_method': 'pattern_based'
            }
            
    except (ValueError, AttributeError):
        pass
    
    # Complete failure
    return {
        'function_name': '',
        'parameters': {},
        'raw_function': response,
        'extracted_successfully': False,
        'extraction_method': 'failed',
        'error': 'No extractable function found'
    }

def compare_parameters(predicted_params, ground_truth_params):
    """Compare predicted parameters with ground truth"""
    if not isinstance(predicted_params, dict) or not isinstance(ground_truth_params, dict):
        return False
    
    # Check if all ground truth parameters are present and correct
    for key, value in ground_truth_params.items():
        if key not in predicted_params:
            return False
        if predicted_params[key] != value:
            return False
    
    return True

def parse_test_sample(text):
    """Parse test sample to extract components"""
    # Extract patient_id, query, and ground truth from formatted text
    patient_id_match = re.search(r'Patient ID: (\w+)', text)
    query_match = re.search(r'Pateint Query: (.*?)\n\n### Response:', text, re.DOTALL)
    ground_truth_match = re.search(r'### Response:\n(.*?)(?:<\|eot_id\||$)', text, re.DOTALL)
    
    if not all([patient_id_match, query_match, ground_truth_match]):
        return None
        
    return {
        'patient_id': patient_id_match.group(1),
        'query': query_match.group(1).strip(),
        'ground_truth_response': ground_truth_match.group(1).strip()
    }

def test_model_on_sample(patient_id, query, max_length=1024):
    """Test model on a single sample"""
    messages = [
        {
            "role": "system",
            "content": "You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function."
        },
        {
            "role": "user",
            "content": f"Patient ID: {patient_id}\nQuery: {query}"
        }
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.1,              
            top_p=0.95,                 
            repetition_penalty=1.0,     
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response_tokens = outputs[0][input_length:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=False)
    response = response.replace(tokenizer.eos_token, "").strip()
    
    return response

print("✅ Utility functions defined!")

✅ Utility functions defined!


In [7]:
# Cell 5: Generate Predictions
print("🧪 Starting prediction generation...")
results = []

for i, sample in enumerate(test_dataset):
    # Parse sample
    parsed = parse_test_sample(sample['text'])
    if not parsed:
        print(f"⚠️ Skipping sample {i}: Could not parse")
        continue
    
    patient_id = parsed['patient_id']
    query = parsed['query']
    ground_truth_response = parsed['ground_truth_response']
    
    # Get the type from the dataset (this is what you mentioned exists)
    sample_type = sample.get('type', 'unknown')  # Use existing type field
    
    # Get model prediction
    predicted_response = test_model_on_sample(patient_id, query)
    
    # Extract functions
    predicted_function = extract_function_from_response(predicted_response)
    ground_truth_function = extract_function_from_response(ground_truth_response)
    
    # Store results
    result = {
        'sample_id': i,
        'patient_id': patient_id,
        'query': query,
        'type': sample_type,  # Use the existing type from dataset
        'ground_truth_response': ground_truth_response,
        'predicted_response': predicted_response,
        'ground_truth_function_name': ground_truth_function['function_name'],
        'predicted_function_name': predicted_function['function_name'],
        'ground_truth_parameters': ground_truth_function['parameters'],
        'predicted_parameters': predicted_function['parameters'],
        'function_extracted': predicted_function['extracted_successfully'],
        'function_name_correct': predicted_function['function_name'] == ground_truth_function['function_name'],
        'parameters_correct': compare_parameters(
            predicted_function['parameters'], 
            ground_truth_function['parameters']
        ),
        'extraction_method': predicted_function.get('extraction_method', 'unknown')
    }
    
    results.append(result)
    
    if (i + 1) % 5 == 0:
        print(f"✅ Processed {i + 1}/{len(test_dataset)} samples...")

print(f"🎯 Prediction generation complete! Processed {len(results)} samples")

🧪 Starting prediction generation...
✅ Processed 5/765 samples...
✅ Processed 10/765 samples...
✅ Processed 15/765 samples...
✅ Processed 20/765 samples...
✅ Processed 25/765 samples...
✅ Processed 30/765 samples...
✅ Processed 35/765 samples...
✅ Processed 40/765 samples...
✅ Processed 45/765 samples...
✅ Processed 50/765 samples...
✅ Processed 55/765 samples...
✅ Processed 60/765 samples...
✅ Processed 65/765 samples...
✅ Processed 70/765 samples...
✅ Processed 75/765 samples...
✅ Processed 80/765 samples...
✅ Processed 85/765 samples...
✅ Processed 90/765 samples...
✅ Processed 95/765 samples...
✅ Processed 100/765 samples...
✅ Processed 105/765 samples...
✅ Processed 110/765 samples...
✅ Processed 115/765 samples...
✅ Processed 120/765 samples...
✅ Processed 125/765 samples...
✅ Processed 130/765 samples...
✅ Processed 135/765 samples...
✅ Processed 140/765 samples...
✅ Processed 145/765 samples...
✅ Processed 150/765 samples...
✅ Processed 155/765 samples...
✅ Processed 160/765 sam

In [8]:
# Cell 6: Save Results
# Create output directory
output_dir = "/home/jovyan/Works/MSc_Thesis/mh_function_calling/model_evaluation/comprehensive_evaluation"
os.makedirs(output_dir, exist_ok=True)

# Save results as CSV
results_df = pd.DataFrame(results)
results_file = f"{output_dir}/fc_predictions_and_ground_truth.csv"
results_df.to_csv(results_file, index=False)

# Save detailed results as JSON for complex data
results_json_file = f"{output_dir}/fc_detailed_results.json"
with open(results_json_file, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"✅ Saved results to: {results_file}")
print(f"✅ Saved detailed results to: {results_json_file}")
print(f"📊 Dataset summary:")
print(f"   Total samples: {len(results)}")
print(f"   Types: {results_df['type'].value_counts().to_dict()}")
print(f"   Extraction success rate: {results_df['function_extracted'].mean():.3f}")
print("🎉 Prediction generation complete!")

✅ Saved results to: /home/jovyan/Works/MSc_Thesis/mh_function_calling/model_evaluation/comprehensive_evaluation/fc_predictions_and_ground_truth.csv
✅ Saved detailed results to: /home/jovyan/Works/MSc_Thesis/mh_function_calling/model_evaluation/comprehensive_evaluation/fc_detailed_results.json
📊 Dataset summary:
   Total samples: 765
   Types: {'explicit': 192, 'behavioral': 159, 'metaphorical': 143, 'symptom': 143, 'implicit': 128}
   Extraction success rate: 1.000
🎉 Prediction generation complete!
